# Inference verification
Check the exported model with the same Predictor used by FastAPI. Scores are not calibrated disease probabilities.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
from ml.inference.predictor import Predictor
from PIL import Image
import pandas as pd
checkpoint = Path("artifacts/model/model.pt")
if checkpoint.exists():
    predictor = Predictor(checkpoint)
    row = pd.read_csv(config["manifest"]).query("split == 'val'").iloc[0]
    with Image.open(Path(config["dataset_root"]) / row.path) as image:
        result, _, _ = predictor.predict(image)
    assert abs(sum(result["probabilities"].values()) - 1) < 1e-6
    display(result)
else:
    print("Model unavailable: complete approved training before research inference.")